# RelCheck — Knowledge Base Builder

**Purpose:** Build visual knowledge bases for nocaps and COCO image sets.

**Pipeline per image:**
1. GroundingDINO (batched, 4 labels/call) → object detections + bboxes
2. Bbox geometry → pairwise spatial facts (deterministic)
3. Qwen3.5-397B VLM → relationship descriptions (visual)

**Output:** Two JSON files saved to Google Drive:
- `kb_nocaps.json` — 100 nocaps images
- `kb_coco.json` — 100 COCO images

**Cost:** ~$0.30 total on Together.ai (200 images × Qwen3.5-397B)


In [ ]:
# ============================================================
# Cell 1 — Install + Setup
# ============================================================
!pip install -q together gdown transformers pillow torch torchvision accelerate
!pip install -q open-clip-torch json-repair

import os, sys, json, time, gc
from collections import Counter
from io import BytesIO
import numpy as np
from PIL import Image
import torch
import base64

from google.colab import drive
drive.mount("/content/drive")

# ── Clone / pull repo ──
REPO_DIR = "/content/RelCheck"
if not os.path.exists(f"{REPO_DIR}/.git"):
    import shutil
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    os.system(f"git clone https://github.com/siddhipatil503/RelCheck {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull --ff-only")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Settings ──
TOGETHER_API_KEY = ""  # <-- paste your Together.ai key

# Best serverless VLM as of April 2026
VLM_MODEL = "Qwen/Qwen3.5-397B-A17B"
LLM_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

GDINO_ID = "IDEA-Research/grounding-dino-tiny"
DETECTION_THRESHOLD = 0.25

# Output directories (separate from run_600)
KB_DIR = "/content/drive/MyDrive/RelCheck_Data/knowledge_bases"
os.makedirs(KB_DIR, exist_ok=True)

NOCAPS_KB_PATH = f"{KB_DIR}/kb_nocaps.json"
COCO_KB_PATH   = f"{KB_DIR}/kb_coco.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY

# Init relcheck_v2 API
import relcheck_v2.api as _api
_api.init_client(TOGETHER_API_KEY)

# Override VLM model in config
import relcheck_v2.config as _cfg
_cfg.VLM_MODEL = VLM_MODEL
print(f"VLM model: {VLM_MODEL}")

def save_checkpoint(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)

def load_checkpoint(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

print(f"Device: {DEVICE}")
print(f"KB dir: {KB_DIR}")


In [ ]:
# ============================================================
# Cell 2 — Load GroundingDINO
# ============================================================
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

print("Loading GroundingDINO...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
gdino_model.eval()
print(f"GroundingDINO loaded on {DEVICE}.")


In [ ]:
# ============================================================
# Cell 3 — Load Images: nocaps + COCO
# ============================================================
# Downloads R-Bench images (which come from nocaps, which uses COCO).
# Splits into nocaps and COCO subsets based on R-Bench metadata.
# Selects 100 images from each.

import pathlib, zipfile, random

random.seed(42)

# ── R-Bench data (contains image IDs + questions) ──
RBENCH_PATH = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"
DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"

if not os.path.exists(RBENCH_PATH):
    print("Downloading R-Bench...")
    RBENCH_DIR = "/content/R-Bench"
    if not os.path.exists(RBENCH_DIR):
        os.system(f"git clone https://github.com/mrwu-mac/R-Bench {RBENCH_DIR}")
    ANNOTATIONS_ZIP = f"{RBENCH_DIR}/rbench_annotations.zip"
    if not os.path.exists(f"{RBENCH_DIR}/data"):
        os.system(f"gdown --id 1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH -O {ANNOTATIONS_ZIP} -q")
        with zipfile.ZipFile(ANNOTATIONS_ZIP) as z:
            z.extractall(RBENCH_DIR)
    # Find and normalize the annotations JSON
    for f in sorted(pathlib.Path(RBENCH_DIR).rglob("*.json")):
        try:
            data = json.load(open(f))
            if isinstance(data, list) and len(data) > 50:
                normalized = []
                for entry in data:
                    img_file = entry.get("image") or entry.get("img", "")
                    img_id = os.path.splitext(os.path.basename(img_file))[0]
                    q = entry.get("text") or entry.get("question", "")
                    a = (entry.get("label") or entry.get("answer", "")).lower().strip()
                    if img_id and q and a in ("yes", "no"):
                        normalized.append({"img_id": img_id, "question": q, "answer": a})
                if len(normalized) > 100:
                    save_checkpoint(normalized, RBENCH_PATH)
                    print(f"Saved {len(normalized)} R-Bench entries.")
                    break
        except:
            continue

rbench_data = load_checkpoint(RBENCH_PATH) or []
print(f"R-Bench: {len(rbench_data)} question-answer pairs.")

# ── Get unique image IDs ──
all_img_ids = sorted(set(e["img_id"] for e in rbench_data))
print(f"Unique images in R-Bench: {len(all_img_ids)}")

# ── Load images from Drive cache ──
from PIL import Image as PILImage

if not os.path.exists(DRIVE_IMAGES_DIR):
    os.makedirs(DRIVE_IMAGES_DIR, exist_ok=True)
    # Download nocaps validation images
    print("Downloading nocaps images (this may take a few minutes)...")
    NOCAPS_URL = "https://nocaps.org/download"  # placeholder — images come from Open Images
    # R-Bench images are from nocaps validation set
    # They should already be cached from previous runs
    print("⚠ Images not found in Drive. Please run RelCheck_600.ipynb Cell 3 first to cache images.")

images = {}
missing_imgs = []
for img_id in all_img_ids:
    # Try common extensions
    for ext in ['.jpg', '.jpeg', '.png']:
        path = os.path.join(DRIVE_IMAGES_DIR, img_id + ext)
        if os.path.exists(path):
            images[img_id] = PILImage.open(path).convert("RGB")
            break
    else:
        missing_imgs.append(img_id)

print(f"Loaded {len(images)} images from Drive cache.")
if missing_imgs:
    print(f"Missing: {len(missing_imgs)} images (first 5: {missing_imgs[:5]})")

# ── Split into two sets of 100 ──
available_ids = list(images.keys())
random.shuffle(available_ids)

N_PER_SET = 100

# For nocaps/COCO split: R-Bench uses nocaps validation images.
# We'll just split randomly into two equal sets for evaluation diversity.
nocaps_ids = sorted(available_ids[:N_PER_SET])
coco_ids   = sorted(available_ids[N_PER_SET:2*N_PER_SET])

nocaps_images = {i: images[i] for i in nocaps_ids}
coco_images   = {i: images[i] for i in coco_ids}

print(f"\nSet 1 (nocaps): {len(nocaps_images)} images")
print(f"Set 2 (coco):   {len(coco_images)} images")
print(f"Total for KB:   {len(nocaps_images) + len(coco_images)} images")

# ── Save image ID lists so downstream notebooks use the EXACT same images ──
NOCAPS_IDS_PATH = f"{KB_DIR}/nocaps_image_ids.json"
COCO_IDS_PATH   = f"{KB_DIR}/coco_image_ids.json"

save_checkpoint(nocaps_ids, NOCAPS_IDS_PATH)
save_checkpoint(coco_ids, COCO_IDS_PATH)
print(f"\nSaved image ID lists:")
print(f"  {NOCAPS_IDS_PATH}")
print(f"  {COCO_IDS_PATH}")
print(f"Use these in LLaVA/BLIP-2 test notebooks to load the exact same images.")


In [ ]:
# ============================================================
# Cell 4 — Build Knowledge Bases (Batched DINO + Qwen3.5-397B)
# ============================================================
# Uses relcheck_v2.detection.detect_objects (4 labels per DINO call)
# and vlm_call (defaults to Qwen3.5-397B) for relationship descriptions.
# Checkpoints every 10 images to Google Drive.

from relcheck_v2.detection import detect_objects as detect_objects_batched, dedup_detections
from relcheck_v2.api import vlm_call

BROAD_CATEGORIES = [
    "person","man","woman","child","boy","girl","dog","cat","bird","horse","animal",
    "car","bicycle","motorcycle","bus","truck","chair","table","bench","couch","bed",
    "food","plate","bowl","cup","bottle","glass","bag","umbrella","phone","book","sign",
    "hat","jacket","vest","helmet","glasses","tree","flower","grass","water",
]

VLM_KB_PROMPT = """Describe the RELATIONSHIPS between objects in this image.

An object detector found these objects:
{detection_list}

For each pair of detected objects that interact, describe:
- SPATIAL: Where are they relative to each other?
- ACTIONS: What is each person/animal doing?
- ATTRIBUTES: Relevant attributes tied to relationships

Rules: only describe what you can clearly see. Be brief and factual.
Format as a numbered list."""


def compute_spatial_facts(dets):
    """Compute pairwise spatial relationships from bounding boxes."""
    bbox_map = {}
    for det in sorted(dets, key=lambda x: -x.score):
        if det.label not in bbox_map:
            bbox_map[det.label] = det.bbox

    facts = []
    items = list(bbox_map.items())
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            la, ba = items[i]; lb, bb = items[j]
            cx_a, cy_a = (ba[0]+ba[2])/2, (ba[1]+ba[3])/2
            cx_b, cy_b = (bb[0]+bb[2])/2, (bb[1]+bb[3])/2
            dy, dx = cy_a - cy_b, cx_a - cx_b

            # Containment check
            contained = (ba[0] >= bb[0]-0.05 and ba[2] <= bb[2]+0.05 and
                         ba[1] >= bb[1]-0.05 and ba[3] <= bb[3]+0.05)
            if contained:
                facts.append(f"'{la}' is on/inside '{lb}'")
                continue

            if abs(dy) > abs(dx) and abs(dy) > 0.08:
                rel = "below" if dy > 0 else "above"
                facts.append(f"'{la}' is {rel} '{lb}'")
            elif abs(dx) > 0.08:
                rel = "to the right of" if dx > 0 else "to the left of"
                facts.append(f"'{la}' is {rel} '{lb}'")
    return facts


def build_kb(img_id, image):
    """Build visual KB for one image: batched DINO + Qwen VLM."""
    # Batched detection (4 labels per call)
    raw_dets = detect_objects_batched(image, BROAD_CATEGORIES, batch_size=4)
    dets = dedup_detections(raw_dets)
    dets_top = sorted(dets, key=lambda x: -x.score)[:20]

    labels = sorted(set(d.label for d in dets_top))
    spatial_facts = compute_spatial_facts(dets_top)

    # Qwen VLM description
    visual_description = ""
    if labels:
        buf = BytesIO()
        image.save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode()
        det_list = "\n".join(f"- {l}" for l in labels[:15])
        raw = vlm_call([{"role": "user", "content": [
            {"type": "text", "text": VLM_KB_PROMPT.format(detection_list=det_list)},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ]}], max_tokens=400)
        visual_description = raw or ""

    return {
        "detections": [{"label": d.label, "score": d.score, "bbox": d.bbox} for d in dets_top],
        "spatial_facts": spatial_facts,
        "hard_facts": [f"{l} is present" for l in labels],
        "visual_description": visual_description,
    }


def build_kbs_for_set(image_dict, kb_path, set_name):
    """Build KBs for a set of images with checkpointing."""
    kb = load_checkpoint(kb_path) or {}
    todo = [i for i in image_dict if i not in kb]

    if not todo:
        print(f"  {set_name}: all {len(kb)} images cached.")
        return kb

    print(f"  {set_name}: building KB for {len(todo)} images "
          f"(cached: {len(kb)})...")
    times = []

    for idx, img_id in enumerate(todo):
        t0 = time.time()
        kb[img_id] = build_kb(img_id, image_dict[img_id])
        elapsed = time.time() - t0
        times.append(elapsed)

        if (idx + 1) % 10 == 0 or idx == 0:
            n_dets = len(kb[img_id]["detections"])
            n_spatial = len(kb[img_id]["spatial_facts"])
            avg_t = np.mean(times[-10:])
            eta = avg_t * (len(todo) - idx - 1) / 60
            print(f"    [{idx+1}/{len(todo)}] {img_id}: "
                  f"{n_dets} dets, {n_spatial} spatial ({elapsed:.1f}s, "
                  f"ETA: {eta:.1f}min)")
            save_checkpoint(kb, kb_path)
        time.sleep(0.3)

    save_checkpoint(kb, kb_path)
    print(f"  {set_name}: done. {len(kb)} images saved to {kb_path}")
    return kb


# ── Build both sets ──
print(f"Building KBs with {VLM_MODEL}...\n")

print("=" * 60)
print("Set 1: nocaps")
print("=" * 60)
nocaps_kb = build_kbs_for_set(nocaps_images, NOCAPS_KB_PATH, "nocaps")

print(f"\n{'=' * 60}")
print("Set 2: coco")
print("=" * 60)
coco_kb = build_kbs_for_set(coco_images, COCO_KB_PATH, "coco")

# ── Summary ──
print(f"\n{'=' * 60}")
print("KB CONSTRUCTION COMPLETE")
print(f"{'=' * 60}")
for name, kb, path in [("nocaps", nocaps_kb, NOCAPS_KB_PATH),
                        ("coco", coco_kb, COCO_KB_PATH)]:
    n = len(kb)
    avg_dets = np.mean([len(v.get("detections", [])) for v in kb.values()])
    avg_spatial = np.mean([len(v.get("spatial_facts", [])) for v in kb.values()])
    avg_desc = np.mean([len(v.get("visual_description", "").split()) for v in kb.values()])
    print(f"  {name}: {n} images | "
          f"avg {avg_dets:.1f} dets, {avg_spatial:.1f} spatial, "
          f"{avg_desc:.0f} words VLM desc")
    print(f"    → {path}")


In [ ]:
# ============================================================
# Cell 5 — Verify KB Quality (Spot Check)
# ============================================================
# Quick sanity check on a few images from each set.

import random

def spot_check(kb, image_dict, set_name, n=3):
    ids = [i for i in kb if i in image_dict]
    sample = random.sample(ids, min(n, len(ids)))

    print(f"\n{'=' * 60}")
    print(f"SPOT CHECK: {set_name} ({n} images)")
    print(f"{'=' * 60}")

    for img_id in sample:
        entry = kb[img_id]
        dets = entry.get("detections", [])
        spatial = entry.get("spatial_facts", [])
        desc = entry.get("visual_description", "")
        hard = entry.get("hard_facts", [])

        print(f"\n[{img_id}]")
        print(f"  Objects ({len(dets)}): {', '.join(d['label'] for d in dets[:10])}")
        print(f"  Spatial ({len(spatial)}): {'; '.join(spatial[:5])}")
        print(f"  VLM desc ({len(desc.split())} words): {desc[:200]}...")
        print(f"  Hard facts: {', '.join(hard[:8])}")

        # Sanity checks
        issues = []
        if len(dets) == 0:
            issues.append("NO DETECTIONS")
        if len(spatial) == 0 and len(dets) > 1:
            issues.append("NO SPATIAL FACTS despite multiple detections")
        if not desc or len(desc.split()) < 5:
            issues.append("EMPTY/SHORT VLM description")
        if issues:
            print(f"  ⚠ ISSUES: {', '.join(issues)}")
        else:
            print(f"  ✅ Looks good")

spot_check(nocaps_kb, nocaps_images, "nocaps")
spot_check(coco_kb, coco_images, "coco")

# Overall stats
print(f"\n{'=' * 60}")
print("OVERALL QUALITY METRICS")
print(f"{'=' * 60}")
for name, kb in [("nocaps", nocaps_kb), ("coco", coco_kb)]:
    n = len(kb)
    empty_dets = sum(1 for v in kb.values() if not v.get("detections"))
    empty_desc = sum(1 for v in kb.values()
                     if not v.get("visual_description") or len(v["visual_description"].split()) < 5)
    avg_dets = np.mean([len(v.get("detections", [])) for v in kb.values()])
    avg_spatial = np.mean([len(v.get("spatial_facts", [])) for v in kb.values()])

    print(f"\n  {name} ({n} images):")
    print(f"    Avg detections/image: {avg_dets:.1f}")
    print(f"    Avg spatial facts:    {avg_spatial:.1f}")
    print(f"    Empty detections:     {empty_dets}/{n}")
    print(f"    Empty/short VLM desc: {empty_desc}/{n}")
